CALCULAR 
- Población en edad de trabajar
- población que no está en edad de trabajar
- Población económicamente activa
- Población económicamente inactiva
- Tasa de ocupación
- Tasa de desempleo

In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
#POBLACION EN EDAD DE TRABAJAR 
PET = ft[ft['PET']==1]['FEX_C18'].sum()
PET

In [0]:
#POBLACION QUE NO ESTA EN EDAD DE TRABAJAR
NO_PET = cgse[(cgse['P6040'] < 15)]['FEX_C18'].sum()
NO_PET

In [0]:
#POBLACION ECONOMICAMENTE ACTIVA 
PEA = ft[ft['FT']==1]['FEX_C18'].sum()
PEA

In [0]:
#POBLACION ECONOMICAMENTE INACTIVA 
PEI = ft[ft['PET']==1 & (ft['P6240'].isin([3, 4, 5, 6]))]['FEX_C18'].sum()
PEI

In [0]:
#TASA DE DESOCUPACION
D = desempleados[desempleados['DSI'] == 1]['FEX_C18'].sum()
TD = (D / PEA)*100
print(f'Tasa de desocupacion:  {TD:.2f} %')

In [0]:
#TASA DE OCUPACION
O = PEA - D
TO = (O/PET)*100
print(f'Tasa de ocupacion:  {TO:.2f} %')



2.TASA DE DESEMPLEO PARA HOMBRES Y MUJERES EN TODOS LOS DEPARTAMENTOS

In [0]:
import pandas as pd
#Primero hay que anexar P3271 a la tabla de desempleados y fuerza de trabajo
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')
ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
                       on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')                        

In [0]:
# desempleados_dpto -> cantidad de hombres y mujeres desempleados por departamento
desempleados_dpto_genero = desempleados[desempleados['DSI']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

In [0]:
desempleados_dpto_genero

In [0]:
#cantidad de hombres y mujeres economicamente activos por departamento
pea_dpto_genero = ft[ft['FT']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

In [0]:
pea_dpto_genero

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero, 'DESEMPLEADOS':desempleados_dpto_genero})
tabla_desempleo

In [0]:
tabla_desempleo['TD']= (tabla_desempleo['DESEMPLEADOS']/tabla_desempleo['PEA'])*100
tabla_desempleo.head(100)

In [0]:
# 4. Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}

sexo_nombre = {1:'Hombre',2:'Mujer'}

tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo_nombre)

In [0]:
tabla_desempleo = tabla_desempleo[['Departamento','Sexo','PEA','DESEMPLEADOS','TD']]
tabla_desempleo

In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(by=[ 'Sexo','TD'], ascending=False)
tabla_desempleo
#DONDE SE VEN MAYORES TASAS DE DESEMPLEO SON EN LOS DEPARTAMENTOS PUTUMAYO, SUCRE, VIHADA, SAN ANDRES Y BOLIVAR CON EL DISTINTIVO DE QUE SON DE SEXO MUJER
#LOS DEPARTAMENTOS CON MENORES TASAS DE DESEMPLEO SON: VAUPES, CAUCA, RISARALDA, CESAR Y NARIÑO; CON LA CARACTERISTICA DE QUE LOS HOMBRES TIENEN LA MENOR TASA DE DESEMPLEO 

3.PARA CADA UNO DE LOS DEPARTAMENTOS ¿CUAL ES EL GRUPO DE EDAD CON MAYOR Y MENOR TASA DE DESEMPLEO?

In [0]:
import numpy as np

condiciones = [(desempleados['P6040'] >= 18) & (desempleados['P6040'] < 24),(desempleados['P6040'] >= 24) & (desempleados['P6040'] < 30),(desempleados['P6040'] >= 30) & (desempleados['P6040'] < 40),(desempleados['P6040'] >= 40) & (desempleados['P6040'] < 50),(desempleados['P6040'] >= 60)]

categorias = ['18 a 24 años','24 a 30 años','30 a 40 años','40 a 50 años','60 años en adelante']
desempleados['GRUPO_EDAD'] = np.select(condiciones, categorias, default='Fuera de los grupos')

desempleados = desempleados[desempleados['GRUPO_EDAD'] != 'Fuera de los grupos']

condiciones = [(ft['P6040'] >= 18) & (ft['P6040'] < 24),(ft['P6040'] >= 24) & (ft['P6040'] < 30),(ft['P6040'] >= 30) & (ft['P6040'] < 40),(ft['P6040'] >= 40) & (ft['P6040'] < 50),(ft['P6040'] >= 60)]

categorias = ['18 a 24 años','24 a 30 años','30 a 40 años','40 a 50 años','60 años en adelante']
ft['GRUPO_EDAD'] = np.select(condiciones, categorias, default='Fuera de los grupos')

ft = ft[ft['GRUPO_EDAD'] != 'Fuera de los grupos']


In [0]:
desempleados

In [0]:
#DESEMPLEO POR DEPARTAMENTO Y POR CATEGORIA DE EDAD
desempleados_edad = desempleados[desempleados['DSI']==1].groupby(['GRUPO_EDAD','DPTO'])['FEX_C18'].sum()

In [0]:
desempleados_edad 

In [0]:
#PEA POR GRUPO DE EDAD
pea_edad = ft[ft['FT']==1].groupby(['GRUPO_EDAD','DPTO'])['FEX_C18'].sum()

In [0]:
pea_edad

In [0]:
tabla_desempleados_edad = pd.DataFrame({'PEA': pea_edad, 'DESEMPLEADOS': desempleados_edad})

In [0]:
tabla_desempleados_edad

In [0]:
tabla_desempleados_edad['TD']= (tabla_desempleados_edad['DESEMPLEADOS']/tabla_desempleados_edad['PEA'])*100

In [0]:
tabla_desempleados_edad

In [0]:
#DEPARTAMENTOS CON MAYOR TASA DE DESEMPLEO POR CATEGORIA DE EDAD 
dpto = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}

tabla_desempleados_edad = tabla_desempleados_edad.reset_index()

tabla_desempleados_edad['Departamento'] = tabla_desempleados_edad['DPTO'].map(dpto)

In [0]:
tabla_desempleados_edad

In [0]:
#GRUPO DE EDAD CON MAYOR TASA DE DESEMPLEO
tabla_desempleados_edad = tabla_desempleados_edad.sort_values(by='TD', ascending=False)
tabla_desempleados_edad
#EL GRUPO CON MAYOR TASA DE DESEMPLEO EN ENTRE 18 A 24 AÑOS EN LOS DEPARTAMENTOS DE SAN ANDRES, SUCRE, VICHADA, BOLIVAR Y PUTUMAYO; EL ULTIMO CON EN GRUPO DE EDAD DE 24 A 30 AÑOS

In [0]:
#GRUPO DE EDAD CON MENOR TASA DE DESEMPLEO 
tabla_desempleados_edad = tabla_desempleados_edad.sort_values(by='TD', ascending=True)
tabla_desempleados_edad
#EL GRUPO CON MENOR TASA DE DESEMPLEO ES DE 60 AÑOS EN ADELANTE EN LOS DEPARTEMENTOS DE RISARALDA, META, CHOCO, CAUCA Y MAGDALENA 

4.¿CUALES SON LOS 3 DEPARTAMENTOS CON MENOR TASA DE DESEMPLEO DE MUJERES ENTRE 18 A 24 AÑOS?

In [0]:
condiciones = [(desempleados['P6040'] >= 18) & (desempleados['P6040'] <= 25)]

categorias = ['18 a 25 años']

desempleados['GRUPO_EDAD_MUJERES'] = np.select( condiciones, categorias, default='Fuera de los grupos')

desempleados = desempleados[ desempleados['GRUPO_EDAD_MUJERES'] != 'Fuera de los grupos'].copy()


condiciones = [ (ft['P6040'] >= 18) & (ft['P6040'] <= 25)]
categorias = ['18 a 25 años']

ft['GRUPO_EDAD_MUJERES'] = np.select( condiciones, categorias, default='Fuera de los grupos')

ft = ft[ft['GRUPO_EDAD_MUJERES'] != 'Fuera de los grupos'].copy()

In [0]:
desempleados_mujeres_18_25 = desempleados[desempleados['GRUPO_EDAD_MUJERES']=='18 a 25 años'].groupby(['GRUPO_EDAD_MUJERES','DPTO'])['FEX_C18'].sum() 

In [0]:
desempleados_mujeres_18_25

In [0]:
pea_mujeres_18_25 = ft[ft['GRUPO_EDAD_MUJERES']=='18 a 25 años'].groupby(['GRUPO_EDAD_MUJERES','DPTO'])['FEX_C18'].sum()
pea_mujeres_18_25

In [0]:
tabla_desempleados_mujeres_18_25= pd.DataFrame({'PEA':pea_mujeres_18_25,'DESEMPLEADOS':desempleados_mujeres_18_25})
tabla_desempleados_mujeres_18_25

In [0]:
tabla_desempleados_mujeres_18_25['TD'] = (tabla_desempleados_mujeres_18_25['DESEMPLEADOS']/tabla_desempleados_mujeres_18_25['PEA'])*100
tabla_desempleados_mujeres_18_25 

In [0]:
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}
tabla_desempleados_mujeres_18_25 = tabla_desempleados_mujeres_18_25.reset_index()
tabla_desempleados_mujeres_18_25 = tabla_desempleados_mujeres_18_25.reset_index()
tabla_desempleados_mujeres_18_25['DPTO'] = tabla_desempleados_mujeres_18_25['DPTO'].map(dpto_nombre)
tabla_desempleados_mujeres_18_25

In [0]:
#MAYOR TASA DE DESEMPLEO EN MUJERES ENTRE 18 A 25 AÑOS
tabla_desempleados_mujeres_18_25 = tabla_desempleados_mujeres_18_25.sort_values(by='TD', ascending=False)
tabla_desempleados_mujeres_18_25
#LOS DEPARTAMENTOS CON MENOR TASA DE DESEMPLEO EN MUJERES ENTRE LOS 18 Y 25 AÑOS SON: GUAVIARE, NARIÑO Y CAUCA 

5.INGLABO (PROMEDIO DE INGRESOS LABORALES) PARA LOS DEPARTAMENTOS DE RISARALDA, CALDAS, QUINDIO, ANTIOQUIA Y VALLE DEL CAUCA 

In [0]:
ocupados

In [0]:
departamentos = [66, 17, 63, 5, 76]

filtro = ocupados[ocupados['DPTO'].isin(departamentos)]

promedio = filtro['INGLABO'].mean()

print(f"El promedio de ingresos laborales es: ${promedio:,.2f}")
print(filtro.groupby('DPTO')['INGLABO'].mean())